# Projeto CardioIA - Fase 2: Diagnóstico Automatizado
**Desenvolvedor: Kleber Foks** | RM: 562225 | Turma: 2TIAOA
---
Este Notebook contém o desafio completo da Fase 2 integrando a extração de linguagem natural (baseado em regras) e o módulo de classificação de Machine Learning para a triagem.

## Parte 1: Extração de Informações (Módulo Especialista)
Lê descrições brutas e extrai sintomas para provisão de diagnóstico baseado em um banco de regras do CSV.

In [ ]:
import csv

def carregar_mapa(caminho_csv):
    """
    Lê o arquivo CSV e retorna uma lista de dicionários com as regras (Sintoma 1, Sintoma 2 -> Doença).
    """
    mapa = []
    try:
        with open(caminho_csv, mode='r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for linha in reader:
                mapa.append({
                    'sintoma1': linha['Sintoma 1'].strip().lower(),
                    'sintoma2': linha['Sintoma 2'].strip().lower(),
                    'doenca': linha['Doença Associada'].strip()
                })
    except FileNotFoundError:
        print(f"Erro: O arquivo {caminho_csv} não foi encontrado.")
    return mapa

def carregar_frases(caminho_txt):
    """
    Lê o arquivo TXT com os relatos dos pacientes ignorando linhas em branco.
    """
    try:
        with open(caminho_txt, mode='r', encoding='utf-8') as f:
            # Retorna uma lista limpando espaços vazios
            return [linha.strip() for linha in f if linha.strip()]
    except FileNotFoundError:
        print(f"Erro: O arquivo {caminho_txt} não foi encontrado.")
        return []

def analisar(frases, mapa):
    """
    Analisa os sintomas de cada frase em base comparativa com o mapa (Rule-based NLP).
    """
    print("-------------------------------------------------------------------------")
    print("       DIAGNÓSTICO AUTOMATIZADO - MÓDULO INTELIGENTE (FASE 2)          ")
    print("-------------------------------------------------------------------------\n")

    for i, frase in enumerate(frases, 1):
        print(f"[Paciente {i} Relato] -> \"{frase}\"")
        frase_lower = frase.lower()
        
        diagnosticos_encontrados = set()
        sintomas_detectados = set()

        for regra in mapa:
            # Procuramos se a string da frase possui os sintomas-chave mapeados
            s1_presente = regra['sintoma1'] in frase_lower
            s2_presente = regra['sintoma2'] in frase_lower
            
            if s1_presente: 
                sintomas_detectados.add(regra['sintoma1'])
            if s2_presente: 
                sintomas_detectados.add(regra['sintoma2'])
            
            # Condição Básica para a Onotologia Médica do Exercício:
            # Se o sintoma 1 OU o sintoma 2 estiverem na frase, levanta o alerta da respectiva doença.
            if s1_presente or s2_presente:
                diagnosticos_encontrados.add(regra['doenca'])

        # Se a IA por regras mapeou alguma coisa...
        if diagnosticos_encontrados:
            print(f" -> Sintomas identificados: {', '.join(sintomas_detectados)}")
            print(f" -> Possível Diagnóstico Assistido: {', '.join(diagnosticos_encontrados)}\n")
        else:
            print(" -> Sintomas identificados: Nenhum mapeado na ontologia.")
            print(" -> Possível Diagnóstico Assistido: Encaminhar para triagem manual médica (Fora do Mapa).\n")


if __name__ == "__main__":
    # Carregando conhecimento
    mapa_db = carregar_mapa("mapa_conhecimento.csv")
    frases_relatadas = carregar_frases("sintomas.txt")
    
    # Processando
    if mapa_db and frases_relatadas:
        analisar(frases_relatadas, mapa_db)



## Parte 2: Classificador Básico de Texto (Machine Learning)
Aplica processamento avançado PNL (TF-IDF Vectorizer) e uma Regressão Logística para predizer níveis de risco no hospital.

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

def main():
    print("-------------------------------------------------------------------------")
    print("     TREINAMENTO IA: CLASSIFICADOR DE RISCO (NLP + TF-IDF) - FASE 2      ")
    print("-------------------------------------------------------------------------\n")

    # 1. Carregar base de dados
    try:
        df = pd.read_csv("dataset_risco.csv", encoding="utf-8")
        print(f"[OK] Base de Dados Carregada: {len(df)} registros encontrados.")
    except Exception as e:
        print(f"[ERRO] Falha ao carregar CSV: {e}")
        return
        
    print("\n[Amostra dos Dados de Triagem Inicial]")
    print(df.head(3))
    
    # 2. Separar as features (x = frases) e os alvos (y = situação)
    X = df['frase']
    y = df['situacao'].str.strip() # Limpa os possiveis espacos
    
    # 3. Processamento de Linguagem Natural (TF-IDF Vectorizer)
    # Transforma texto em vetores de importância matemática com base na frequência das palavras
    vetorizador = TfidfVectorizer()
    X_vetorizado = vetorizador.fit_transform(X)
    
    # Exibir quantas colunas (palavras exclusivas) nosso vocabulário tem
    print(f"\n[TF-IDF] Vocabulário montado com {X_vetorizado.shape[1]} palavras únicas.")
    
    # 4. Dividir Base em Treino e Teste
    # Pegamos 30% pro teste como validação e 70% pro computador aprender
    X_treino, X_teste, y_treino, y_teste = train_test_split(X_vetorizado, y, test_size=0.3, random_state=42)
    
    # 5. Treinando o Classificador de Machine Learning
    print("\n[Treinamento] Construindo o modelo Logistic Regression (Scikit-Learn)...")
    
    # Usando Logist Regression devido ao dataset ser bem pequeno e puramente textual (Binario).
    # Caso prefira arvore de decisao, descomente a linha abaixo e comente a de cima!
    modelo = LogisticRegression(random_state=42)
    # modelo = DecisionTreeClassifier(random_state=42) 
    
    modelo.fit(X_treino, y_treino)
    
    # 6. Previsões e Métricas
    previsoes = modelo.predict(X_teste)
    acuracia = accuracy_score(y_teste, previsoes)
    
    print(f"\n[RESULTADOS DO MODELO DE TRIAGEM]")
    print(f" -> Acurácia Geral do Sistema: {acuracia * 100:.2f}%")
    
    print("\n[Relatório de Classificação Detalhado]")
    print(classification_report(y_teste, previsoes, zero_division=0))
    
    # 7. Simulador Interativo: Testando com frases nunca antes vistas pela IA
    frases_invisiveis_ao_treino = [
        "Sinto muito peso no meu peito, dor forte que está esmagando meu braço.",
        "Meu joelho está raspado depois de um tombo de bicicleta que levei ontem.",
        "To com uma falta de ar subita na sala.",
        "dor passageira no punho digitando muito"
    ]
    
    print("\n------------------------------------------------------------------")
    print("[Inferência (Real-Time)] - Testando em Frases Ocultas Inéditas")
    print("------------------------------------------------------------------")
    
    # O pipeline obriga passarmos os dados novos pelo mesmo filtro TF-IDF
    vetores_novos = vetorizador.transform(frases_invisiveis_ao_treino)
    diagnosticos = modelo.predict(vetores_novos)
    
    for frase, classe in zip(frases_invisiveis_ao_treino, diagnosticos):
        print(f"Frase: '{frase}'\n   [SISTEMA APONTA: {classe.upper()}]\n")
        
if __name__ == "__main__":
    main()



### ⚠️ Laboratório de Viés e Governança de Dados (Bônus)
Conforme as orientações sobre a responsabilidade do uso das IAs, este bloco aprofunda a avaliação. Ele treina o modelo antigo ao lado de um novo modelo com base expandida para comprovar na tela as correções de Falso Positivo (Viés).

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

def treinar_modelo(caminho_csv):
    try:
        df = pd.read_csv(caminho_csv, encoding='utf-8')
    except Exception as e:
        print(f"Erro ao ler {caminho_csv}: {e}")
        return None, None
        
    X = df['frase']
    y = df['situacao'].str.strip()
    
    vetorizador = TfidfVectorizer()
    # Adquirindo vocabulário
    X_vet = vetorizador.fit_transform(X)
    
    modelo = LogisticRegression(random_state=42)
    modelo.fit(X_vet, y)
    
    # Acurácia no próprio dataset apenas para validar se aprendeu e os pesos não estão zerados
    acc = accuracy_score(y, modelo.predict(X_vet))
    
    return vetorizador, modelo, acc, len(df)

def main():
    print("-------------------------------------------------------------------------")
    print("      LABORATÓRIO: O IMPACTO DO VIÉS DE DADOS (DATA GOVERNANCE)          ")
    print("-------------------------------------------------------------------------\n")
    
    print("[1] Treinando Inteligência A: Base de Dados Pobre (Problema Antigo)")
    vet_ruim, mod_ruim, acc_ruim, len_ruim = treinar_modelo("dataset_risco.csv")
    print(f" -> Modelo treinado com apenas {len_ruim} linhas.")
    
    print("\n[2] Treinando Inteligência B: Base de Dados Expandida (Governança Aplicada)")
    vet_bom, mod_bom, acc_bom, len_bom = treinar_modelo("dataset_risco_expandido.csv")
    print(f" -> Modelo treinado com {len_bom} linhas mais detalhadas.")
    
    print("\n=========================================================================")
    print("   DESAFIO PRÁTICO: O PACIENTE FOI PRA TRIAGEM DO HOSPITAL               ")
    print("=========================================================================\n")
    
    frases_teste = [
        "Sinto muito peso no meu peito, a dor está esmagando meu braço.",
        "Meu joelho está ralado depois de um tombo de bicicleta que levei ontem.",
        "To com uma falta de ar subita na sala.",
        "dor passageira no punho de tanto digitar no pc velho."
    ]
    
    for frase in frases_teste:
        print(f"Paciente chega e diz: \"{frase}\"")
        
        # Testando no modelo pobre
        vet1 = vet_ruim.transform([frase])
        pred1 = mod_ruim.predict(vet1)[0]
        
        # Testando no modelo enriquecido
        vet2 = vet_bom.transform([frase])
        pred2 = mod_bom.predict(vet2)[0]
        
        # Conferindo se eles discordam
        if pred1 == pred2:
            print(f"   [Ambos os Modelos Concordaram] -> {pred2.upper()}\n")
        else:
            print(f"   -> DIVERGÊNCIA IDENTIFICADA!")
            print(f"      - A Inteligência Pobre (Viés) apontou: {pred1.upper()}")
            print(f"      - A Inteligência Maior corrigiu para : {pred2.upper()}\n")

if __name__ == "__main__":
    main()

